# 06 — Draft Approval & Scheduling (Phase 1a)

Review, edit, and approve LLM-generated drafts before publishing.

**Workflow:**
1. Load pending drafts from `content_queue.json`
2. Review each draft — edit content if needed
3. Set `scheduled_at` datetime for each approved draft
4. Write updated queue back to state
5. Then run notebook 05 to publish approved & scheduled drafts

In [1]:
import sys
sys.path.insert(0, '..')

from agent.storage import LocalStorage
from agent.models import ContentQueue, Draft

store = LocalStorage(base_dir='../state')
queue_data = store.read('content_queue.json')

if not queue_data:
    raise RuntimeError('No content_queue.json — run notebook 03 first.')

queue = ContentQueue(**queue_data)
print(f'Pending:   {len(queue.drafts)}')
print(f'Approved:  {len(queue.approved)}')
print(f'Published: {len(queue.published)}')
print(f'Rejected:  {len(queue.rejected)}')

Pending:   3
Approved:  0
Published: 0
Rejected:  0


## 1. Review Pending Drafts

Displays all drafts with `status: pending_approval`.

In [2]:
for i, draft in enumerate(queue.drafts):
    print(f'=== Draft {i} | {draft.channel} ({draft.language}) ===')
    print(f'ID:     {draft.id}')
    print(f'Source: {draft.source_blog_post}')
    print(f'Link:   {draft.link}')
    print(f'Chars:  {len(draft.content)}')
    print(f'---')
    print(draft.content)
    print()

=== Draft 0 | mastodon (en) ===
ID:     draft_cb352481
Source: Independent x402 facilitator on Optimism and Base
Link:   https://www.fretchen.eu/x402/?utm_source=mastodon&utm_campaign=growth-agent
Chars:  297
---
Can decentralized platforms truly scale? One key insight from our latest article is that independent x402 facilitators are crucial for Optimism and Base to reach their full potential. Read more: https://www.fretchen.eu/x402/?utm_source=mastodon&utm_campaign=growth-agent #blockchain #Web3 #scaling

=== Draft 1 | bluesky (en) ===
ID:     draft_9a82a5a4
Source: Independent x402 facilitator on Optimism and Base
Link:   https://www.fretchen.eu/x402/?utm_source=bluesky&utm_campaign=growth-agent
Chars:  173
---
"Unlocking growth on Optimism and Base: get the inside scoop from an independent x402 facilitator https://www.fretchen.eu/x402/?utm_source=bluesky&utm_campaign=growth-agent"

=== Draft 2 | mastodon (de) ===
ID:     draft_8e4b825f
Source: Independent x402 facilitator on Optimis

## 2. Edit Drafts

Edit the content of individual drafts below. Change the index and new text
as needed. Run this cell multiple times for different drafts.

**Tip:** Copy the draft content from above, paste below, and edit.

In [7]:
# === EDIT THIS CELL ===
# Change DRAFT_INDEX to the draft number from the list above (0, 1, 2, ...)
DRAFT_INDEX = 0

# Replace the content below with your edited version:
NEW_CONTENT = """
The internet was built for humans.

Logins, subscriptions, payment forms.

But what happens when the primary users are AI agents?

I’ve been exploring x402 — a simple idea with big implications:
→ APIs return “402 Payment Required”
→ agents pay automatically
→ access is unlocked

No accounts. No subscriptions. Just machine-to-machine transactions.

This flips the model:
From SaaS → to pay-per-call infrastructure.

Feels like a missing primitive for agent economies.

Curious if this becomes:
- the “HTTP moment” for AI services
- or just another crypto experiment

Wrote a short exploration:
https://www.fretchen.eu/x402/?utm_source=mastodon&utm_campaign=growth-agent
""".strip()

# --- Apply edit ---
if NEW_CONTENT != 'Your edited post content here.':
    draft = queue.drafts[DRAFT_INDEX]
    old_len = len(draft.content)
    draft.content = NEW_CONTENT
    print(f'Updated draft {DRAFT_INDEX} ({draft.channel}/{draft.language})')
    print(f'  {old_len} chars → {len(draft.content)} chars')
    print(f'  Preview: {draft.content[:100]}...')
else:
    print('No edit applied — replace NEW_CONTENT with your edited text.')

Updated draft 0 (mastodon/en)
  0 chars → 671 chars
  Preview: The internet was built for humans.

Logins, subscriptions, payment forms.

But what happens when the...


## 3. Approve or Reject Drafts

Only drafts listed in `APPROVE` will be publishable. Everything else stays pending.
Time is optional — defaults to one post per day starting tomorrow 09:00 UTC.

In [8]:
from datetime import datetime, timezone, timedelta

# === LIST DRAFTS TO APPROVE (by index from the review above) ===
APPROVE = [0]         # e.g. [0, 1, 2]
REJECT  = []          # e.g. [2]
# Everything else stays as pending_approval.

# === OPTIONAL: override scheduled time for specific drafts ===
# Default: one per day starting tomorrow 09:00 UTC
# SCHEDULE = { 0: datetime(2026, 4, 15, 12, 0, tzinfo=timezone.utc) }
SCHEDULE = {}

# --- Compute defaults ---
base = datetime.now(timezone.utc).replace(hour=9, minute=0, second=0, microsecond=0) + timedelta(days=1)
for rank, idx in enumerate(APPROVE):
    if idx not in SCHEDULE:
        SCHEDULE[idx] = base + timedelta(days=rank)

# --- Show plan ---
for i, draft in enumerate(queue.drafts):
    if i in APPROVE:
        t = SCHEDULE[i].strftime('%a %d %b %H:%M UTC')
        print(f'  ✅ Draft {i} | {draft.channel} ({draft.language}) → approve → 📅 {t}')
    elif i in REJECT:
        print(f'  ❌ Draft {i} | {draft.channel} ({draft.language}) → reject')
    else:
        print(f'  ⏸️  Draft {i} | {draft.channel} ({draft.language}) → stays pending')
    print(f'     {draft.content[:80]}...')
    print()

  ✅ Draft 0 | mastodon (en) → approve → 📅 Tue 14 Apr 09:00 UTC
     The internet was built for humans.

Logins, subscriptions, payment forms.

But w...

  ⏸️  Draft 1 | bluesky (en) → stays pending
     "Unlocking growth on Optimism and Base: get the inside scoop from an independent...

  ⏸️  Draft 2 | mastodon (de) → stays pending
     Kann ein unabhängiger Facilitator wirklich den Erfolg von Optimism und Base ents...



## 4. Apply Decisions

Run this cell to move drafts to approved/rejected based on the schedule above.

In [9]:
# Apply decisions (process in reverse so indices stay valid)
for idx in sorted(set(APPROVE) | set(REJECT), reverse=True):
    draft = queue.drafts[idx]

    if idx in REJECT:
        draft.status = 'rejected'
        queue.rejected.append(draft)
        queue.drafts.pop(idx)
        print(f'❌ Rejected draft {idx} ({draft.channel}/{draft.language})')
    elif idx in APPROVE:
        draft.status = 'approved'
        draft.scheduled_at = SCHEDULE[idx]
        queue.approved.append(draft)
        queue.drafts.pop(idx)
        t = SCHEDULE[idx].strftime('%a %d %b %H:%M UTC')
        print(f'✅ Approved draft {idx} ({draft.channel}/{draft.language}) → {t}')

print(f'\nRemaining pending: {len(queue.drafts)}')
print(f'Approved queue:    {len(queue.approved)}')
print(f'Rejected:          {len(queue.rejected)}')

✅ Approved draft 0 (mastodon/en) → Tue 14 Apr 09:00 UTC

Remaining pending: 2
Approved queue:    1
Rejected:          0


## 5. Save Updated Queue

In [10]:
store.write('content_queue.json', queue)
print('✅ content_queue.json saved.')
print(f'   Pending:   {len(queue.drafts)}')
print(f'   Approved:  {len(queue.approved)}')
print(f'   Published: {len(queue.published)}')
print(f'   Rejected:  {len(queue.rejected)}')
print()
for d in queue.approved:
    sched = d.scheduled_at.strftime('%a %d %b %H:%M UTC') if d.scheduled_at else 'no time set'
    print(f'   📅 {d.id} | {d.channel} ({d.language}) | {sched}')

✅ content_queue.json saved.
   Pending:   2
   Approved:  1
   Published: 0
   Rejected:  0

   📅 draft_cb352481 | mastodon (en) | Tue 14 Apr 09:00 UTC


## Next Steps

Run **notebook 05** to publish approved drafts where `scheduled_at <= now`.